# Aula 02B · Diferenças finitas

**Nenhum conceito novo hoje.** O encontro inteiro é laboratório sobre o
[capítulo 2](https://lacouth.github.io/metodos_telecom-site/unidade2-derivadas/02-diferencas-finitas/): a pergunta que ficou do encontro A, dois 🎯, o
problema da maré resolvido e a [Lista 02](https://lacouth.github.io/metodos_telecom-site/listas/lista02/) começada em sala.

**Roteiro** (1h40): 🔥 aquecimento · ⚠️ a regra do encontro · 🎯 prática · 🧩 a
maré · 📋 a lista · 🚪

## Como usar este caderno

- **Rode a célula ⚙️** logo abaixo antes de tudo (e de novo se o Colab reiniciar).
- Nenhum conceito novo hoje: tudo o que você precisa está no encontro A e no
  capítulo (links 📖).
- Em cada **🎯**, escreva a função no lugar de `# sua solução aqui` e rode a
  célula `confere`. Tente antes de abrir a 💡 Dica.

In [ ]:
# ⚙️ Rode esta célula antes de tudo. Ela prepara a correção automática dos
# exercícios 🎯 — não precisa ler (usa coisas que não fazem parte do curso).
import math


def _mostra(argumentos):
    textos = []
    for a in argumentos:
        textos.append(a.__name__ if callable(a) else repr(a))
    return ", ".join(textos)


def _numero(x):
    try:
        float(x)
        return not isinstance(x, (str, bool))
    except (TypeError, ValueError):
        return False


def _igual(veio, esperado, tol):
    # Número: compara com tolerância relativa, porque conta com float quase
    # nunca bate na última casa. Lista, tupla ou array: item a item.
    if _numero(esperado) and _numero(veio):
        return math.isclose(float(veio), float(esperado), rel_tol=tol, abs_tol=tol)
    if isinstance(esperado, (list, tuple)) and hasattr(veio, "__len__") and not isinstance(veio, str):
        if len(veio) != len(esperado):
            return False
        return all(_igual(v, e, tol) for v, e in zip(veio, esperado))
    return veio == esperado


def confere(funcao, casos, tol=1e-6):
    """Chama funcao com cada caso (argumentos, esperado) e diz se acertou."""
    certos = 0
    for numero, (argumentos, esperado) in enumerate(casos, start=1):
        chamada = f"{funcao.__name__}({_mostra(argumentos)})"
        try:
            veio = funcao(*argumentos)
        except Exception as erro:
            print(f"❌ {chamada} deu erro: {type(erro).__name__}: {erro}")
            continue
        if veio is not None and _igual(veio, esperado, tol):
            certos += 1
            print(f"✅ {chamada} devolveu {veio!r}")
        elif veio is None:
            print(f"❌ {chamada} devolveu None — faltou o return?")
        else:
            print(f"❌ {chamada} devolveu {veio!r}, mas devia ser {esperado!r}")
    print(f"{certos} de {len(casos)} certos")


def confere_valor(nome, valor, esperado, tol=1e-6):
    """Diz se a variável `nome` ficou com o valor esperado."""
    if valor is not None and _igual(valor, esperado, tol):
        print(f"✅ {nome} = {valor!r}")
    else:
        print(f"❌ {nome} vale {valor!r}, mas devia ser {esperado!r}")

# --- bibliotecas desta aula ---
import numpy as np
import matplotlib.pyplot as plt

## 🔥 Aquecimento — a pergunta que ficou

A central usa $f(x+h)$ e $f(x-h)$. Se a função é uma **tabela de medições** — a
temperatura a cada 5 minutos —, que $h$ você usa? E no **primeiro** ponto da
tabela, onde não existe $f(x-h)$?

Discuta com o colega do lado antes de abrir.

<details>
<summary><b>▶ Resposta</b></summary>

O $h$ **não é escolhido**: é a distância entre as medições (5 minutos). E no
primeiro ponto a central não existe — usa-se a **progressiva**, que só precisa do
vizinho da direita (no último ponto, a regressiva). O preço é um erro maior nas
bordas. É exatamente o assunto da semana que vem, o [capítulo 3](https://lacouth.github.io/metodos_telecom-site/unidade2-derivadas/03-derivada-de-dados/).

</details>

## ⚠️ A regra do encontro

**Derivada numérica sem conferência não vale.** Antes de confiar num número,
faça pelo menos uma destas: compare com a derivada exata, quando existir; ou
recalcule com $h/2$ e veja se o resultado quase não muda. Um número que muda
muito quando $h$ muda é um número em que você **não** pode confiar.

## 🎯 Prática

Vem do bloco *3. Três fórmulas, um circuito* do encontro A.
📖 [capítulo 2 · Três fórmulas, um circuito](https://lacouth.github.io/metodos_telecom-site/unidade2-derivadas/02-diferencas-finitas/#tres-formulas-um-circuito)

### 🎯 Sua vez — A regressiva

Escreva `regressiva(f, x, h)`, que devolve $\dfrac{f(x) - f(x-h)}{h}$.

In [ ]:
def regressiva(f, x, h):
    # sua solução aqui
    pass

In [ ]:
def cubo(x):
    return x**3


confere(regressiva, [
    ((cubo, 2, 0.1), 11.410000000000009),
    ((cubo, 2, 1), 7.0),
])

<details>
<summary><b>💡 Dica</b></summary>

Igual à progressiva, mas o vizinho é `x - h` e a ordem da subtração se inverte.

</details>

Vem do bloco *7. Mesmo método, outra área* do encontro A.
📖 [capítulo 2 · Mesmo método, outra área](https://lacouth.github.io/metodos_telecom-site/unidade2-derivadas/02-diferencas-finitas/#mesmo-metodo-outra-area)

**Física do dia a dia.** Um café servido a 90 °C numa sala a 25 °C esfria segundo
a lei de Newton. A célula 📦 abaixo já tem a temperatura `T(t)`, com $t$ em minutos.

In [ ]:
# 📦 dados prontos — só rode esta célula
# Temperatura (°C) de uma xícara de café, t minutos depois de servida,
# numa sala a 25 °C (lei de resfriamento de Newton).
def T(t):
    return 25 + 65 * np.exp(-0.08 * t)

### 🎯 Sua vez — O café esfriando

Escreva `taxa_cafe(t)`, que devolve a taxa de variação da temperatura do
café (°C por minuto) no minuto `t`, pela central com `h = 0.01`.

Depois de passar: o café esfria mais depressa no começo ou no fim? Por quê?

In [ ]:
def taxa_cafe(t):
    # sua solução aqui
    pass

In [ ]:
confere(taxa_cafe, [
    ((0,), np.float64(-5.200000554666673)),
    ((10,), np.float64(-2.3365108626371978)),
])

<details>
<summary><b>💡 Dica</b></summary>

A mesma fórmula do passo 4 do encontro A, com `T` no lugar de `i` e `h = 0.01` dentro da função.

</details>

## 🧩 Resolvendo o problema

> *"**No horário dos passeios, entre 6 h e 18 h, quando a maré sobe mais depressa,
> e quantos centímetros por hora ela sobe nesse momento?**"* — o barqueiro do
> Picãozinho.

A tábua de marés vira a função `mare(t)` na célula 📦 abaixo (nível em metros,
$t$ em horas depois da meia-noite).

In [ ]:
# 📦 dados prontos — só rode esta célula
# Nível da maré (m) em João Pessoa, t horas depois da meia-noite.
# Maré semidiurna: período de 12,42 h, preamar (maré cheia) às 3 h.
def mare(t):
    return 1.3 + 1.1 * np.cos(2 * np.pi * (t - 3) / 12.42)

### 🎯 Sua vez — A maré mais rápida

Escreva `maior_subida(horas)`, que recebe uma lista de horários, calcula a
derivada central de `mare` em cada um (com `h = 0.01`) e devolve a tupla
`(hora, taxa)` do horário em que a maré **sobe mais depressa**, com a taxa
em **centímetros por hora**.

É o padrão "extremo": um candidato inicial e um laço que troca quando acha
um melhor.

In [ ]:
def maior_subida(horas):
    # sua solução aqui
    pass

In [ ]:
confere(maior_subida, [
    (([6, 7, 8, 9, 10],), (10, np.float64(21.65265022359364))),
    (([6, 7.5, 9, 10.5, 12, 13.5, 15, 16.5, 18],), (12, np.float64(54.942863115893246))),
])

<details>
<summary><b>💡 Dica</b></summary>

Comece com `melhor_hora = horas[0]` e a taxa dela. No laço, calcule a taxa de
cada `t` e troque os dois quando a taxa for maior. No `return`, multiplique a
taxa por 100 (metros → centímetros).

</details>

Agora responda ao barqueiro, com a sua função, de meia em meia hora entre 6 h e
18 h:

In [ ]:
horas = []
for k in range(25):
    horas.append(6 + k / 2)
print("(hora, cm/h):", maior_subida(horas))

<details>
<summary><b>▶ E a conferência (a regra do encontro)?</b></summary>

A maré é $1{,}3 + 1{,}1\cos\!\big(2\pi(t-3)/12{,}42\big)$. A derivada exata tem
valor máximo $1{,}1 \cdot 2\pi / 12{,}42 \approx 0{,}556$ m/h $= 55{,}6$ cm/h,
quando o seno vale $-1$: $t = 3 + \tfrac{3}{4}\cdot 12{,}42 \approx 12{,}3$ h. A
grade de meia em meia hora acha 12,5 h — perto, mas limitada pela grade. É a
metade do caminho entre a maré baixa (9h13) e a cheia seguinte (15h25): a maré
corre mais no meio da subida. (Por volta da meia-noite há outra subida igual: a
maré de João Pessoa sobe **duas vezes por dia**. Por isso a pergunta fixou o
horário dos passeios.)

Achar o instante **exato** do máximo (onde a segunda derivada zera) é um problema
de raiz: Unidade 3.

</details>

## 📋 A lista, começada aqui

Abra a [Lista 02](https://lacouth.github.io/metodos_telecom-site/listas/lista02/). O **Exercício 01** é à mão (✏️), como a parte em papel
da prova — vamos fazê-lo juntos, no papel. $f(x) = 2x^2 - 5x + 1$, $h = 0{,}5$,
aproximar $f'(2)$.

**a)** Quais valores de $f$ você precisa calcular antes de tudo?

<details>
<summary><b>▶ Resposta</b></summary>

$f(1{,}5)$, $f(2)$ e $f(2{,}5)$. Faça as três primeiro e só depois as fórmulas —
é assim que se evita erro de conta na prova.

</details>

**b)** Uma das três fórmulas vai acertar **exatamente**. Qual, e por quê, antes de fazer a conta?

<details>
<summary><b>▶ Resposta</b></summary>

A central. O erro dela é proporcional a $f'''$, e a terceira derivada de um
polinômio de grau 2 é **zero**. A Taylor responde antes da calculadora.

</details>

Termine o exercício e siga para o **Exercício 02**, que é a sua `central` do encontro A — agora com testes.

## 🚪 Antes de sair

**1.** A tábua de marés só dá o nível de hora em hora, e não a função `mare(t)`.
Como você calcularia a taxa às 12 h? E às 0 h, o primeiro valor da tábua?

<details>
<summary><b>▶ Resposta</b></summary>

Às 12 h, central com os vizinhos das 11 h e das 13 h: $h = 1$ h. Às 0 h, não existe
o vizinho das 23 h do dia anterior (na tábua do dia): progressiva. É o capítulo 3.

</details>

**2.** Você mediu a derivada com `h = 0.01` e com `h = 0.005`, e os resultados diferem na **segunda** casa decimal. Pode confiar em algum dos dois?

<details>
<summary><b>▶ Resposta</b></summary>

Não sem investigar. Se a função é suave, a central com esses $h$ devia concordar em
muitas casas. Uma diferença grande sugere $h$ pequeno demais (arredondamento), uma
função com "degraus" ou dado com ruído — o assunto do capítulo 3.

</details>

## 🏠 Para casa

- Termine a [Lista 02](https://lacouth.github.io/metodos_telecom-site/listas/lista02/).
- Leia o começo do [capítulo 3](https://lacouth.github.io/metodos_telecom-site/unidade2-derivadas/03-derivada-de-dados/): e
  quando não há fórmula, só uma tabela?